# Game manager: the turn-by-turn state machine

**Deliverable:** the game loop with the camera as the source of truth (README: *"Infer the human's
move from the changed squares and the legal moves, and ask the human when unsure"*), and rule
legality checked separately from physical placement (Perception and rules milestone).

```
START -> (human is white) WAIT_HUMAN -> DETECT_MOVE -> VALIDATE -> ROBOT_THINK -> ROBOT_EXECUTE -> VERIFY -> WAIT_HUMAN ...
                                 \-> RECOVER <-/                                        \-> RECOVER
any state -> GAME_OVER
```

The manager owns the *rules board* (`self.board`, positions confirmed by the camera). Hardware and
policy are injected (dependency injection), so the same class runs on the real robot, in Gazebo
and fully in software (`game_loop.ipynb`):

* `vision`: `VisionSource.read() -> BoardObservation`
* `arm`: `ArmInterface` (physical_moves.ipynb)
* `policy`: `MovePolicy.choose_move(board) -> chess.Move` (the teacher policy from
  `gambit_teach/notebooks/move_selection.ipynb`, or `EnginePolicy`)
* `listeners`: callbacks `fn(event: MoveEvent)` for logging, hesitation features, the player model.

**TODO checklist**
- [ ] State enum and transition table are given (the spec); implement `transition` to enforce it.
- [ ] `wait_for_stable_board`, `match_legal_moves` (legal-move matching from a camera observation).
- [ ] One handler per state (`on_start` ... `on_recover`), `step`, `run`.
- [ ] `on_human_timeout`: what to do when the human does not move (prompt? pause clock?).
- [ ] `resolve_ambiguous`: several legal moves fit the observation (e.g. promotion piece unclear).
- [ ] RECOVER branch: depends on `choose_recovery_action` / `plan_correction` (recovery.ipynb).
- [ ] Human-move "done" signal: stable board only, or also a button / clock press?
- [ ] Draw offers, resignation, takebacks (out of scope for Mark-I?).
- [ ] Self-checks (plan at the bottom).

In [ ]:
%run ./engine.ipynb
%run ./physical_moves.ipynb
%run ./recovery.ipynb

In [ ]:
import enum
import time
from dataclasses import dataclass, field
from typing import Callable, Optional, Protocol

import chess

GAME_SETTINGS = GAME_CFG["game"]

## States and allowed transitions

The table is the specification: `GameManager.transition` should refuse anything not listed, so a
bug shows up as an exception instead of a robot doing something odd.

In [ ]:
class State(enum.Enum):
    START = "start"                  # check the initial board
    WAIT_HUMAN = "wait_human"        # wait for a stable, changed board
    DETECT_MOVE = "detect_move"      # observed board -> legal move
    VALIDATE = "validate"            # rule legality, push the human's move
    ROBOT_THINK = "robot_think"      # policy picks a move, plan arm steps
    ROBOT_EXECUTE = "robot_execute"  # run the arm steps
    VERIFY = "verify"                # camera confirms the robot's move
    RECOVER = "recover"              # mismatch handling
    GAME_OVER = "game_over"


TRANSITIONS: dict[State, set[State]] = {
    State.START: {State.WAIT_HUMAN, State.ROBOT_THINK, State.RECOVER},
    State.WAIT_HUMAN: {State.DETECT_MOVE},
    State.DETECT_MOVE: {State.VALIDATE, State.RECOVER},
    State.VALIDATE: {State.ROBOT_THINK, State.RECOVER},
    State.ROBOT_THINK: {State.ROBOT_EXECUTE},
    State.ROBOT_EXECUTE: {State.VERIFY},
    State.VERIFY: {State.WAIT_HUMAN, State.RECOVER},
    State.RECOVER: {State.START, State.WAIT_HUMAN, State.VERIFY, State.ROBOT_EXECUTE, State.DETECT_MOVE},
    State.GAME_OVER: set(),
}
for _s in State:                         # the game can end from anywhere
    if _s is not State.GAME_OVER:
        TRANSITIONS[_s].add(State.GAME_OVER)

## Injected interfaces

In [ ]:
class VisionSource(Protocol):
    def read(self) -> BoardObservation: ...


class MovePolicy(Protocol):
    def choose_move(self, board: chess.Board) -> chess.Move: ...


class EnginePolicy:
    """Plain strength-limited engine play (the teacher's fallback)."""

    def __init__(self, engine: EngineLike):
        self.engine = engine

    def choose_move(self, board: chess.Board) -> chess.Move:
        """Delegate to the engine's play()."""
        # TODO: implement
        raise NotImplementedError


@dataclass
class MoveEvent:
    ply: int
    by: str                          # "human" or "robot"
    fen_before: str
    move: chess.Move
    san: str
    think_time_s: float
    robot_attempts: int = 0
    plan: str = ""
    observation: Optional[BoardObservation] = None
    extra: dict = field(default_factory=dict)

## Reading the board

`wait_for_stable_board`: the board is trusted only when the same placement is seen in
`stable_frames` consecutive frames with no hand over the board.

`match_legal_moves`: try every legal move and keep those whose resulting placement agrees with the
observation. Squares below the confidence threshold are ignored, so one uncertain square does not
hide an otherwise clear move (but an ignored square should never *create* a match on its own: at
least the move's own from/to squares must be confident).

In [ ]:
class HumanTimeout(Exception):
    pass


def wait_for_stable_board(vision: VisionSource, n_frames: int = GAME_SETTINGS["stable_frames"],
                          timeout_s: float = GAME_SETTINGS["human_move_timeout_s"],
                          clock: Callable[[], float] = time.monotonic,
                          max_reads: int = 100_000) -> BoardObservation:
    """Read frames until `n_frames` consecutive hand-free frames show the same placement.

    Raises HumanTimeout after `timeout_s` (measured with the injected `clock`) or `max_reads`.
    """
    # Design:
    # - Keep the last candidate observation and a run-length counter.
    # - A frame with hand_present resets the run (a hand over the board means "not done yet").
    # - Same placement as the candidate -> count up; different -> it becomes the new candidate.
    # - The clock is injected so tests and the simulator are deterministic; max_reads guards
    #   against an infinite loop with a fake clock that never advances.
    # TODO: implement
    raise NotImplementedError


def match_legal_moves(board: chess.Board, obs: BoardObservation,
                      min_conf: float = MIN_SQUARE_CONFIDENCE) -> list[chess.Move]:
    """Every legal move whose resulting placement agrees with the observation."""
    # Design:
    # - For each move in board.legal_moves, compute the board after it (on a copy) and compare
    #   piece maps square by square with obs.piece_map().
    # - Ignore low-confidence squares in the comparison, but skip moves whose own from/to square
    #   is low-confidence (an ignored square must not create the match).
    # - Why compare whole positions: castling, en passant and promotion need no special cases.
    # Decisions for you: is brute force over ~30 legal moves fast enough? (Measure.)
    # TODO: implement
    raise NotImplementedError

## The manager

In [ ]:
class GameManager:
    def __init__(self, vision: VisionSource, arm: ArmInterface, policy: MovePolicy,
                 human_color: chess.Color = chess.WHITE, board: Optional[chess.Board] = None,
                 listeners: Optional[list] = None, clock: Callable[[], float] = time.monotonic,
                 settings: dict = GAME_SETTINGS, recovery_cfg: dict = RECOVERY_CFG):
        """Store the injected dependencies and set up per-game and per-move state."""
        # Design: besides storing the arguments (board defaults to a new chess.Board, listeners to
        # []), the manager needs:
        # - game state: graveyard (physical_moves.Graveyard), state = START, trace = [START];
        # - per-move scratch: observation, pending_move, plan (list[Step]), expected_after (board
        #   the camera should see after the robot's move), recovery_attempts, robot_attempts,
        #   turn_started (clock time the current turn began), robot_think_s, mismatch_human_turn.
        # TODO: implement
        raise NotImplementedError

    # -- plumbing ----------------------------------------------------------------
    def transition(self, new: State) -> None:
        """Move to `new` if TRANSITIONS allows it, else RuntimeError; record it in the trace."""
        # TODO: implement
        raise NotImplementedError

    def emit(self, event: MoveEvent) -> None:
        """Call every listener with the event."""
        # TODO: implement
        raise NotImplementedError

    def human_to_move(self) -> bool:
        # TODO: compare board.turn with human_color
        raise NotImplementedError

    def _after_confirmed_move(self) -> None:
        """Common bookkeeping after a move is confirmed: reset counters, restart the turn clock,
        then GAME_OVER (board.is_game_over(claim_draw=True)) or whoever moves next."""
        # TODO: implement
        raise NotImplementedError

    # -- handlers ----------------------------------------------------------------
    def on_start(self):
        """Check the physical start position against self.board."""
        # Design: wait for a stable board; any diff -> RECOVER (remember the observation and that it
        # is not the human's mismatch); otherwise start the turn clock and go to whoever moves.
        # TODO: implement
        raise NotImplementedError

    def on_wait_human(self):
        """Wait until a stable board differs from the rules board, then DETECT_MOVE."""
        # Design: loop on wait_for_stable_board with the human timeout; on HumanTimeout call
        # on_human_timeout; an unchanged stable board means "not moved yet", keep waiting.
        # TODO: implement
        raise NotImplementedError

    def on_human_timeout(self):
        """What to do when the human has not moved for human_move_timeout_s."""
        # Decisions for you: prompt the human (sound/LED)? log a hesitation event? keep waiting
        # forever or pause the game?
        # TODO: implement
        raise NotImplementedError

    def on_detect_move(self):
        """Observation -> one legal move (VALIDATE) or a mismatch (RECOVER)."""
        # Design: match_legal_moves; exactly one -> pending_move; several -> resolve_ambiguous;
        # none -> RECOVER with the human's mismatch flag and expected board = current board.
        # TODO: implement
        raise NotImplementedError

    def resolve_ambiguous(self, moves: list[chess.Move]) -> chess.Move:
        """Several legal moves fit the observation: pick one or ask."""
        # Decisions for you: re-read the board a few times; if still ambiguous ask the human.
        # TODO: implement
        raise NotImplementedError

    def on_validate(self):
        """Rule legality (separately from placement), then commit the human's move."""
        # Design:
        # - Illegal -> RECOVER. Legal -> build a MoveEvent (think time = observation timestamp
        #   minus turn start, never negative), update the graveyard, push the move, emit.
        # - Then reset counters/turn clock and go to GAME_OVER or ROBOT_THINK.
        # TODO: implement
        raise NotImplementedError

    def _graveyard_after(self, move: chess.Move) -> Graveyard:
        """Track what the human's move put into the graveyard (the human hands captures to it)."""
        # Design: reuse plan_physical_move on a graveyard copy; a MissingPieceError is fine here
        # (the human brought their own promotion piece).
        # TODO: implement
        raise NotImplementedError

    def on_robot_think(self):
        """Ask the policy for a move, time it, plan the arm steps and the expected board."""
        # Design: plan on a graveyard copy (commit only after VERIFY); expected_after = board + move.
        # TODO: implement
        raise NotImplementedError

    def on_robot_execute(self):
        """Run the plan on the arm, then VERIFY even after a failure (the camera decides)."""
        # TODO: implement
        raise NotImplementedError

    def on_verify(self):
        """Camera confirms the robot's move: diff vs expected_after -> RECOVER, else commit."""
        # Design: on success emit a robot MoveEvent (think time, attempts, describe_plan), commit
        # the graveyard changes, push the move, then _after_confirmed_move.
        # TODO: implement
        raise NotImplementedError

    def on_recover(self):
        """diff -> classify -> choose_recovery_action -> transition (see recovery.ipynb)."""
        # Design: map each RecoveryAction to a state: ACCEPT -> DETECT_MOVE/VERIFY, REVERIFY ->
        # WAIT_HUMAN/VERIFY, ARM_RETRY -> plan_correction then ROBOT_EXECUTE, ASK_HUMAN -> ask_human
        # then WAIT_HUMAN/VERIFY, ABORT -> GAME_OVER (human vs robot mismatch picks the branch).
        # Count attempts so the retry budget works.
        # TODO: implement
        raise NotImplementedError

    def ask_human(self, diff: BoardDiff, kind: MismatchKind) -> None:
        """Tell the human what is wrong (which squares) and wait for a confirmation signal."""
        # TODO: implement
        raise NotImplementedError

    HANDLERS = {
        State.START: on_start, State.WAIT_HUMAN: on_wait_human, State.DETECT_MOVE: on_detect_move,
        State.VALIDATE: on_validate, State.ROBOT_THINK: on_robot_think,
        State.ROBOT_EXECUTE: on_robot_execute, State.VERIFY: on_verify, State.RECOVER: on_recover,
    }

    def step(self) -> State:
        """Run the handler for the current state once; return the new state."""
        # TODO: implement (dispatch through HANDLERS)
        raise NotImplementedError

    def run(self, max_plies: Optional[int] = None, max_steps: int = 10_000) -> chess.Board:
        """Run until GAME_OVER, or until `max_plies` half-moves have been confirmed."""
        # Design: step in a loop; stop at GAME_OVER, or once board.ply() >= max_plies while waiting
        # for the next move (WAIT_HUMAN / ROBOT_THINK), so the board is in a confirmed state.
        # max_steps guards against a handler that never progresses. Return the rules board.
        # TODO: implement
        raise NotImplementedError

## Self-check (no hardware: a scripted vision source)

In [ ]:
class _ScriptedVision:
    """Test helper: returns the given observations in order, repeating the last one."""

    def __init__(self, observations):
        self.obs, self.i = list(observations), 0

    def read(self):
        # TODO: implement
        raise NotImplementedError

# Self-check plan (write these as asserts once the functions above work):
# - Table sanity: every transition target is a State; GAME_OVER has no outgoing transitions.
# - match_legal_moves: start position, observation after Nf3 -> [g1f3]; an unrelated low-confidence
#   square (h7) is ignored; low confidence on f3 itself -> []; an extra knight on f4 -> [].
# - wait_for_stable_board: frames [start, hand, after, after, after] with n_frames=3 returns
#   `after` and has read all 5 frames.
# - transition guard: from START, transition(VERIFY) raises RuntimeError.
# - Scripted human move: _ScriptedVision([start]*5 + [after Nf3]*5), 4 steps ->
#   trace START, WAIT_HUMAN, DETECT_MOVE, VALIDATE, ROBOT_THINK and move_stack == [g1f3].
# Example shape of a test:
# b = chess.Board(); a = b.copy(); a.push_san("Nf3")
# assert match_legal_moves(b, BoardObservation.from_board(a)) == [chess.Move.from_uci("g1f3")]
print("Self-check: not written yet (see plan above)")